# Policy RAG — LangChain rewrite

End-to-end port of the raw Azure-OpenAI + Chroma notebook to **LangChain**, built for VS Code.

Same outputs as before: embedding view, cosine matrix, ranked retrieval, metadata filtering, chunk-strategy comparison, LLM reranking, and the top-k trade-off.

**Install once (VS Code terminal):**
```bash
pip install -U langchain langchain-openai langchain-chroma chromadb pandas numpy python-dotenv ipykernel
```
Then pick your Python interpreter / kernel in the top-right of this notebook.

## 1. Setup — Azure clients via LangChain

In [1]:
from pathlib import Path
import os, json, re
import pandas as pd
import numpy as np
from dotenv import load_dotenv

from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_MODEL")            # chat/generation deployment
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")  # embedding deployment

assert AZURE_OPENAI_API_KEY,  "Missing AZURE_OPENAI_API_KEY in .env"
assert AZURE_OPENAI_ENDPOINT, "Missing AZURE_OPENAI_ENDPOINT in .env"
assert CHAT_DEPLOYMENT,       "Missing AZURE_OPENAI_MODEL in .env"
assert EMBEDDING_DEPLOYMENT,  "Missing AZURE_OPENAI_EMBEDDING_MODEL in .env"

# One embeddings object + one chat object. LangChain handles batching internally.
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

llm = AzureChatOpenAI(
    azure_deployment=CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

env_df = pd.DataFrame([
    {"Component": "Chat / Generation", "Azure deployment": CHAT_DEPLOYMENT,      "Purpose": "Generate grounded answers / rerank"},
    {"Component": "Embeddings",        "Azure deployment": EMBEDDING_DEPLOYMENT, "Purpose": "Convert text and queries into vectors"},
])
env_df

,Component,Azure deployment,Purpose
0,Chat / Generation,gpt-4.1,Generate grounded answers / rerank
1,Embeddings,text-embedding-3-small,Convert text and queries into vectors


In [2]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

records_by_strategy = {
    s: load_jsonl(ARTIFACT_DIR / f"policy_chunks_{s}.jsonl")
    for s in ["fixed", "sentence", "recursive", "section"]
}

corpus_df = pd.DataFrame([
    {"Strategy": s, "Chunks": len(rows),
     "Avg chars": round(np.mean([len(x["text"]) for x in rows]), 1)}
    for s, rows in records_by_strategy.items()
])
corpus_df

,Strategy,Chunks,Avg chars
0,fixed,16,581.8
1,sentence,17,482.2
2,recursive,17,539.5
3,section,34,214.7


## 2. What does an embedding actually look like?

In [3]:
sample_texts = [
    "MRI scans require prior authorization.",
    "Advanced imaging needs insurer approval.",
    "The member updated a mailing address.",
]

# LangChain: embed_documents returns a list[list[float]]
sample_vectors = embeddings.embed_documents(sample_texts)

embedding_df = pd.DataFrame([
    {
        "Text": text,
        "Vector Dimensions": len(vec),
        "First 6 Values": str([round(v, 4) for v in vec[:6]]),
        "Vector Norm": round(float(np.linalg.norm(vec)), 4),
    }
    for text, vec in zip(sample_texts, sample_vectors)
])
embedding_df

,Text,Vector Dimensions,First 6 Values,Vector Norm
0,MRI scans require prior authorization.,1536,"[-0.0135, 0.0446, 0.0086, -0.0018, -0.0417, 0....",0.9999
1,Advanced imaging needs insurer approval.,1536,"[0.0253, 0.026, 0.0133, 0.0343, -0.0505, 0.0096]",1.0003
2,The member updated a mailing address.,1536,"[-0.0088, -0.0149, 0.0215, 0.0387, -0.0154, 0....",1.0000


An embedding is a long numeric vector representing semantic characteristics of the text. We compare vectors geometrically, not dimension-by-dimension.

### Cosine similarity
- **1.0** → vectors point in almost the same direction.
- **0.0** → weak directional relationship.
- **Negative values** → opposing directions are mathematically possible.

For a Chroma collection configured with cosine distance:

**cosine similarity ≈ 1 − cosine distance**

So in the retrieval tables below:
- **lower distance is better**
- **higher similarity is better**

## 3. Make semantic similarity visible with a small comparison matrix

In [4]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    (0, 1, "Same business meaning, different wording"),
    (0, 2, "Different business meaning"),
    (1, 2, "Different business meaning"),
]

sim_rows = []
for i, j, relationship in pairs:
    sim_rows.append({
        "Text A": sample_texts[i],
        "Text B": sample_texts[j],
        "Expected Relationship": relationship,
        "Cosine Similarity": round(cosine_similarity(sample_vectors[i], sample_vectors[j]), 4),
    })

pd.DataFrame(sim_rows)

,Text A,Text B,Expected Relationship,Cosine Similarity
0,MRI scans require prior authorization.,Advanced imaging needs insurer approval.,"Same business meaning, different wording",0.6396
1,MRI scans require prior authorization.,The member updated a mailing address.,Different business meaning,0.0870
2,Advanced imaging needs insurer approval.,The member updated a mailing address.,Different business meaning,0.1397


> Embeddings let “insurer approval” retrieve “prior authorization” even though the exact terms differ.

## 4. Embed all chunk strategies and persist them in a Chroma vector store

LangChain's `Chroma` wrapper embeds + upserts for you. We share one persistent client so we can cleanly reset each collection on re-runs.

In [5]:
import chromadb

VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_policy_db")
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

META_KEYS = [
    "doc_id", "source_file", "title", "plan_type", "policy_domain",
    "effective_date", "page", "section", "chunk_strategy",
]

def build_vectorstore(strategy, records):
    name = f"policy_{strategy}"
    # reset collection so re-runs are idempotent
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass

    vs = Chroma(
        client=chroma_client,
        collection_name=name,
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"},
    )

    docs = [
        Document(
            page_content=r["text"],
            metadata={k: r[k] for k in META_KEYS},
        )
        for r in records
    ]
    vs.add_documents(docs, ids=[r["chunk_id"] for r in records])
    return vs

vectorstores = {
    strategy: build_vectorstore(strategy, records)
    for strategy, records in records_by_strategy.items()
}

vector_store_df = pd.DataFrame([
    {"Collection": f"policy_{s}", "Strategy": s,
     "Vectors Stored": vs._collection.count()}
    for s, vs in vectorstores.items()
])
vector_store_df

,Collection,Strategy,Vectors Stored
0,policy_fixed,fixed,16
1,policy_sentence,sentence,17
2,policy_recursive,recursive,17
3,policy_section,section,34


## 5. Ranked semantic retrieval — rank, distance, similarity, source and text

In [6]:
def semantic_search(question, vectorstore, top_k=5, where=None):
    """Return list of (Document, distance). Chroma cosine distance => sim = 1 - distance."""
    return vectorstore.similarity_search_with_score(question, k=top_k, filter=where)

def search_to_df(results):
    rows = []
    for rank, (doc, distance) in enumerate(results, start=1):
        meta = doc.metadata
        rows.append({
            "Rank": rank,
            "Cosine Distance ↓": round(distance, 4),
            "Cosine Similarity ↑": round(1 - distance, 4),
            "Document": meta["doc_id"],
            "Plan": meta["plan_type"],
            "Section": meta["section"],
            "Page": meta["page"],
            "Chunk Strategy": meta["chunk_strategy"],
            "Retrieved Text": re.sub(r"\s+", " ", doc.page_content)[:240] + "...",
        })
    return pd.DataFrame(rows)

question = "For Gold PPO, when does physical therapy start requiring authorization?"
results = semantic_search(question, vectorstores["section"], top_k=5)
search_to_df(results)

,Rank,Cosine Distance ↓,Cosine Similarity ↑,Document,Plan,Section,Page,Chunk Strategy,Retrieved Text
0,1,0.1546,0.8454,REHAB-2026,Multiple Plans,SECTION 2: PLAN-SPECIFIC PHYSICAL THERAPY THRE...,1,section,"For Gold PPO members, prior authorization is r..."
1,2,0.3150,0.6850,GOLD-PPO-2026,Gold PPO,SECTION 4: SPECIALIST SERVICES,1,section,Gold PPO members may access participating spec...
2,3,0.3524,0.6476,GOLD-PPO-2026,Gold PPO,SECTION 3: PHYSICAL THERAPY,1,section,The first 10 physical therapy visits in a bene...
3,4,0.3653,0.6347,SILVER-HMO-2026,Silver HMO,SECTION 3: PHYSICAL THERAPY,1,section,The first 6 physical therapy visits in a benef...
4,5,0.4130,0.5870,GOLD-PPO-2026,Gold PPO,DOCUMENT HEADER,1,section,Synthetic training document - no real member d...


### Reading this ranking

**Rank 1** is the nearest vector result. The similarity score is useful for comparison **within the same embedding model and use case**, but it is not a universal confidence percentage.

> **Important:** A high vector similarity means “semantically close,” not “factually correct.”

## 6. Metadata filtering — before and after in one table

Chroma's newer API prefers explicit operators, so we use `{"$eq": ...}`.

In [7]:
query = "When does physical therapy require authorization?"

before = search_to_df(semantic_search(query, vectorstores["section"], top_k=5))
after = search_to_df(
    semantic_search(
        query,
        vectorstores["section"],
        top_k=5,
        where={"plan_type": {"$eq": "Silver HMO"}},
    )
)

before["Search Mode"] = "Semantic only"
after["Search Mode"] = "Semantic + Silver HMO filter"

pd.concat([before, after], ignore_index=True)[
    ["Search Mode", "Rank", "Cosine Similarity ↑", "Document", "Plan", "Section", "Retrieved Text"]
]

,Search Mode,Rank,Cosine Similarity ↑,Document,Plan,Section,Retrieved Text
0,Semantic only,1,0.7278,GOLD-PPO-2026,Gold PPO,SECTION 3: PHYSICAL THERAPY,The first 10 physical therapy visits in a bene...
1,Semantic only,2,0.7169,SILVER-HMO-2026,Silver HMO,SECTION 3: PHYSICAL THERAPY,The first 6 physical therapy visits in a benef...
2,Semantic only,3,0.6478,REHAB-2026,Multiple Plans,SECTION 2: PLAN-SPECIFIC PHYSICAL THERAPY THRE...,"For Gold PPO members, prior authorization is r..."
3,Semantic only,4,0.5495,GOLD-PPO-2026,Gold PPO,SECTION 2: ADVANCED DIAGNOSTIC IMAGING,"Non-emergency outpatient MRI, CT and PET proce..."
4,Semantic only,5,0.5470,REHAB-2026,Multiple Plans,SECTION 3: CONTINUED THERAPY REVIEW,Requests for visits beyond the no-authorizatio...
5,Semantic + Silver HMO filter,1,0.7169,SILVER-HMO-2026,Silver HMO,SECTION 3: PHYSICAL THERAPY,The first 6 physical therapy visits in a benef...
6,Semantic + Silver HMO filter,2,0.5387,SILVER-HMO-2026,Silver HMO,SECTION 2: ADVANCED DIAGNOSTIC IMAGING,"Non-emergency MRI, CT and PET procedures requi..."
7,Semantic + Silver HMO filter,3,0.4713,SILVER-HMO-2026,Silver HMO,SECTION 6: MEMBER RESPONSIBILITY,Authorization or referral approval does not gu...
8,Semantic + Silver HMO filter,4,0.4168,SILVER-HMO-2026,Silver HMO,SECTION 4: SPECIALIST REFERRALS,A referral from the member's assigned primary-...
9,Semantic + Silver HMO filter,5,0.3791,SILVER-HMO-2026,Silver HMO,SECTION 5: CHIROPRACTIC SERVICES,Chiropractic services are limited to 12 covere...


> **Takeaway:** Filtering changes the candidate universe *before* ranking. It stops the search engine from ranking the wrong plan even when its text is highly similar.

## 7. Compare retrieval across four chunking strategies

In [8]:
query = "What information is needed for continued physical therapy authorization?"

strategy_results = []
for strategy, vs in vectorstores.items():
    df = search_to_df(semantic_search(query, vs, top_k=3))
    for _, row in df.iterrows():
        strategy_results.append({
            "Strategy": strategy,
            "Rank": row["Rank"],
            "Similarity": row["Cosine Similarity ↑"],
            "Document": row["Document"],
            "Section": row["Section"],
            "Text": row["Retrieved Text"],
        })

pd.DataFrame(strategy_results)

,Strategy,Rank,Similarity,Document,Section,Text
0,fixed,1,0.6417,REHAB-2026,FIXED_WINDOW,"For Silver HMO members, prior authorization is..."
1,fixed,2,0.5478,SILVER-HMO-2026,FIXED_WINDOW,Emergency imaging is exempt from prior authori...
2,fixed,3,0.5448,GOLD-PPO-2026,FIXED_WINDOW,ion before the service is scheduled. Emergency...
3,sentence,1,0.6381,REHAB-2026,SENTENCE_GROUP,SECTION 3: CONTINUED THERAPY REVIEW Requests f...
4,sentence,2,0.5588,REHAB-2026,SENTENCE_GROUP,Synthetic training document - no real member d...
5,sentence,3,0.5544,SILVER-HMO-2026,SENTENCE_GROUP,Emergency imaging is exempt from prior authori...
6,recursive,1,0.7138,REHAB-2026,RECURSIVE_WINDOW,"embers, prior authorization is required beginn..."
7,recursive,2,0.5737,SILVER-HMO-2026,RECURSIVE_WINDOW,", CT and PET procedures require prior authoriz..."
8,recursive,3,0.5552,GOLD-PPO-2026,RECURSIVE_WINDOW,"ent MRI, CT and PET procedures require prior a..."
9,section,1,0.6636,SILVER-HMO-2026,SECTION 3: PHYSICAL THERAPY,The first 6 physical therapy visits in a benef...


This output is deliberately comparative: the question is unchanged; only chunk boundaries change.

> **Teaching question:** “Which chunking method puts the complete business rule closest to the top?”

## 8. Ranking vs reranking

**Initial ranking** uses vector similarity and is fast.
**Reranking** takes the top candidate chunks and evaluates their relevance more deeply against the actual question.

For training, we use the Azure chat model as a simple cross-encoder-style reranker:
1. Retrieve top 6 by vector similarity.
2. Ask the LLM to score each candidate from 0–100 for direct answer relevance.
3. Sort by rerank score.
4. Compare rank movement.

In [9]:
def rerank_with_llm(question, initial_df):
    candidates = [
        {
            "rank": int(row["Rank"]),
            "chunk_id": f"C{int(row['Rank'])}",
            "document": row["Document"],
            "section": row["Section"],
            "text": row["Retrieved Text"],
        }
        for _, row in initial_df.iterrows()
    ]

    prompt = f"""
You are reranking retrieved healthcare policy chunks for a user question.

QUESTION:
{question}

CANDIDATES:
{json.dumps(candidates, indent=2)}

Score every candidate from 0 to 100 for how directly it contains evidence
needed to answer the question.
Return ONLY a JSON array in this format:
[
  {{"chunk_id":"C1","relevance_score":95,"reason":"..."}}
]
Do not omit any candidate.
"""

    response = llm.invoke([
        SystemMessage(content="You are a precise enterprise search reranker."),
        HumanMessage(content=prompt),
    ])

    text = response.content.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.I | re.S)
    scores = json.loads(text)
    score_map = {x["chunk_id"]: x for x in scores}

    reranked = initial_df.copy()
    reranked["Candidate"] = [f"C{x}" for x in reranked["Rank"]]
    reranked["Rerank Score ↑"] = reranked["Candidate"].map(
        lambda x: score_map.get(x, {}).get("relevance_score", 0)
    )
    reranked["Rerank Reason"] = reranked["Candidate"].map(
        lambda x: score_map.get(x, {}).get("reason", "")
    )

    reranked = reranked.sort_values(
        ["Rerank Score ↑", "Cosine Similarity ↑"], ascending=[False, False]
    ).reset_index(drop=True)

    reranked["Rank After Rerank"] = np.arange(1, len(reranked) + 1)
    reranked["Rank Movement"] = reranked["Rank"] - reranked["Rank After Rerank"]
    return reranked

rerank_question = "What clinical documentation is required when requesting continued physical therapy?"
initial = search_to_df(semantic_search(rerank_question, vectorstores["section"], top_k=6))

initial[["Rank", "Cosine Similarity ↑", "Document", "Section", "Retrieved Text"]].rename(
    columns={"Rank": "Vector Rank"}
)

,Vector Rank,Cosine Similarity ↑,Document,Section,Retrieved Text
0,1,0.5737,SILVER-HMO-2026,SECTION 3: PHYSICAL THERAPY,The first 6 physical therapy visits in a benef...
1,2,0.5712,REHAB-2026,SECTION 4: MEDICAL NECESSITY,Continued therapy must demonstrate measurable ...
2,3,0.5414,REHAB-2026,SECTION 3: CONTINUED THERAPY REVIEW,Requests for visits beyond the no-authorizatio...
3,4,0.5406,REHAB-2026,SECTION 5: DOCUMENTATION,Clinical documentation must support the servic...
4,5,0.5292,GOLD-PPO-2026,SECTION 3: PHYSICAL THERAPY,The first 10 physical therapy visits in a bene...
5,6,0.5070,REHAB-2026,SECTION 1: SCOPE,"This policy covers physical therapy, occupatio..."


In [10]:
reranked = rerank_with_llm(rerank_question, initial)

reranked[
    ["Rank", "Rank After Rerank", "Rank Movement", "Cosine Similarity ↑",
     "Rerank Score ↑", "Document", "Section", "Rerank Reason"]
].rename(columns={"Rank": "Vector Rank"})

,Vector Rank,Rank After Rerank,Rank Movement,Cosine Similarity ↑,Rerank Score ↑,Document,Section,Rerank Reason
0,3,1,2,0.5414,100,REHAB-2026,SECTION 3: CONTINUED THERAPY REVIEW,Explicitly lists required documentation: diagn...
1,1,2,-1,0.5737,90,SILVER-HMO-2026,SECTION 3: PHYSICAL THERAPY,Directly states that continued therapy request...
2,5,3,2,0.5292,85,GOLD-PPO-2026,SECTION 3: PHYSICAL THERAPY,Specifies that authorization requests must inc...
3,2,4,-2,0.5712,60,REHAB-2026,SECTION 4: MEDICAL NECESSITY,Mentions the need to demonstrate measurable fu...
4,4,5,-1,0.5406,40,REHAB-2026,SECTION 5: DOCUMENTATION,States that clinical documentation must suppor...
5,6,6,0,0.5070,20,REHAB-2026,SECTION 1: SCOPE,Only describes the scope of the policy and doe...


### The reranking output

- **Vector Rank** = broad semantic closeness.
- **Rerank Score** = deeper question-to-chunk relevance assessment.
- **Rank Movement > 0** = chunk moved upward after reranking.
- **Rank Movement < 0** = chunk moved downward.

> **Takeaway:** Retrieval finds candidates; reranking decides which candidates deserve the scarce top context positions.

## 9. Top-k — convert the trade-off into a visible summary

In [11]:
rows = []
query = "Gold PPO chiropractic annual visit limit"

for k in [1, 3, 5, 8]:
    results = semantic_search(query, vectorstores["section"], top_k=k)
    df = search_to_df(results)
    rows.append({
        "Top-K": k,
        "Best Similarity": df["Cosine Similarity ↑"].max(),
        "Unique Documents": df["Document"].nunique(),
        "Unique Plans": df["Plan"].nunique(),
        "Retrieved Characters": sum(len(doc.page_content) for doc, _ in results),
        "Top Result": f"{df.iloc[0]['Document']} | {df.iloc[0]['Section']}",
    })

pd.DataFrame(rows)

,Top-K,Best Similarity,Unique Documents,Unique Plans,Retrieved Characters,Top Result
0,1,0.6571,1,1,182,GOLD-PPO-2026 | SECTION 5: CHIROPRACTIC SERVICES
1,3,0.6571,2,2,483,GOLD-PPO-2026 | SECTION 5: CHIROPRACTIC SERVICES
2,5,0.6571,3,3,855,GOLD-PPO-2026 | SECTION 5: CHIROPRACTIC SERVICES
3,8,0.6571,3,3,1540,GOLD-PPO-2026 | SECTION 5: CHIROPRACTIC SERVICES
